In [ ]:
# Cell 1: Environment Setup
import os
import sys

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

# Set directories
if IS_KAGGLE:
    WORKING_DIR = '/kaggle/working'
    RESULTS_DIR = '/kaggle/working/results'
    # Add repository to Python path
    sys.path.insert(0, '/kaggle/input/gdsearch-repository')
else:
    # Running locally
    WORKING_DIR = os.path.abspath('..')
    RESULTS_DIR = os.path.join(WORKING_DIR, 'results')
    # Add parent directory to Python path
    sys.path.insert(0, WORKING_DIR)

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Working directory: {WORKING_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Python path updated: {sys.path[0]}")

In [ ]:
# Cell 2: Install Dependencies (Kaggle Only)
if IS_KAGGLE:
    print("📦 Installing dependencies...")
    
    # Set critical environment variables FIRST to avoid warnings
    import os
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Prevent tokenizer fork warnings
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ['HF_HUB_OFFLINE'] = '0'
    # Suppress CUDA warnings
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    os.environ['GRPC_VERBOSITY'] = 'ERROR'
    os.environ['GLOG_minloglevel'] = '2'
    
    # Upgrade core packages first to avoid compatibility issues
    !pip install -q --upgrade pip setuptools wheel
    
    # FIXED: Install packages with exact compatible versions to avoid conflicts
    # The order matters - install conflicting packages first with specific versions
    !pip install -q --upgrade "fsspec==2025.3.0"  # FIXED: Exact version for gcsfs
    !pip install -q --upgrade "pyarrow>=14.0.0,<20.0.0"  # FIXED: Compatible with cudf-cu12
    !pip install -q --upgrade "rich>=12.4.4,<14"  # FIXED: Compatible with bigframes
    !pip install -q --upgrade "click>=7.0,!=8.3.0"  # FIXED: Compatible with ray
    !pip install -q --upgrade "cryptography>=19.0,<44"  # FIXED: Compatible with pydrive2
    !pip install -q --upgrade "pyOpenSSL>=19.1.0,<=24.2.1"  # FIXED: Compatible with pydrive2
    !pip install -q --upgrade "huggingface_hub>=0.30.0,<1.0"
    !pip install -q --upgrade "protobuf>=3.20.3,<4.0.0"  # FIXED: Compatible with TensorFlow
    
    # Install remaining required packages
    !pip install -q transformers datasets plotly kaleido psutil scipy optuna mlflow
    
    print("✅ All dependencies installed!")
    
    # Verify datasets loading works
    print("\n🔍 Verifying HuggingFace compatibility...")
    try:
        from datasets import load_dataset
        from transformers import AutoTokenizer
        import transformers
        
        # Suppress unnecessary warnings
        transformers.logging.set_verbosity_error()
        import warnings
        warnings.filterwarnings('ignore', message='Some weights.*were not initialized')
        warnings.filterwarnings('ignore', message='.*cuFFT.*')
        warnings.filterwarnings('ignore', message='.*cuDNN.*')
        warnings.filterwarnings('ignore', message='.*cuBLAS.*')
        
        # Quick test
        _ = AutoTokenizer.from_pretrained('distilbert-base-uncased')
        print("✅ HuggingFace models accessible")
    except Exception as e:
        print(f"⚠️  HuggingFace access limited: {e}")
        print("   NLP experiments will use fallback mode (simpler models)")
        print("   This is normal for some Kaggle environments.")
else:
    print("⚠️  Not on Kaggle - assuming dependencies are already installed")
    # Set environment variables for local runs too
    import os
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
# Cell 3: GPU & Environment Check
import torch

print("=" * 60)
print("GPU Configuration")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    GPU_FLAG = "--kaggle-t4"
else:
    print("⚠️ No GPU available - training will be slower")
    GPU_FLAG = ""

print(f"\nPyTorch: {torch.__version__}")
print(f"Python: {sys.version}")

In [ ]:
# Cell 4: Configuration Summary

print("="*60)
print("🚀 PRODUCTION CONFIGURATION")
print("="*60)
print("Seeds: 10 seeds (42,123,456,789,1011,1213,1415,1617,1819,2021)")
print("Quick mode: False (full benchmark)")
print("Resume: True (safe to interrupt)")
print("Hyperparameter tuning: Enabled")
print("Experiments: ALL")
print("Profiling: Enabled")
print("Kaggle T4 optimizations: Enabled")
print("="*60)

In [ ]:
# Cell 5: Run Complete Benchmark Suite
#
# This executes the full production benchmark with:
# ✅ 10 seeds for high statistical power
# ✅ Resume logic (safe to interrupt)
# ✅ Cross-experiment aggregation
# ✅ Statistical analysis (t-tests, effect sizes, power analysis)
# ✅ Hyperparameter tuning with Optuna
# ✅ Interactive visualizations
# ✅ Publication-ready reports
# ✅ Automatic fallback for failed experiments (continues with remaining)
#

import subprocess
import time

# Determine script path based on environment
if IS_KAGGLE:
    # On Kaggle, the repository is in the input dataset
    script_path = "/kaggle/input/gdsearch-repository/run_all_kaggle.py"
else:
    # Running locally
    script_path = os.path.join(WORKING_DIR, "run_all_kaggle.py")

# Build command
cmd = [
    "python", script_path,
    "--experiments", "all",
    "--seeds", "42,123,456,789,1011,1213,1415,1617,1819,2021",
    "--results-dir", RESULTS_DIR,
    "--resume",
    "--profile"
]

# Add Kaggle T4 optimizations if GPU available
if torch.cuda.is_available():
    cmd.append("--kaggle-t4")

print("="*70)
print("🚀 STARTING GDSEARCH BENCHMARK SUITE")
print("="*70)
print(f"\nCommand: {' '.join(cmd)}\n")
print("This will take several hours. The process is resumable - safe to interrupt.")
print("Individual experiment failures will NOT crash the entire pipeline.")
print("="*70 + "\n")

start_time = time.time()

# Execute the benchmark - don't raise on error, just report
result = subprocess.run(cmd, capture_output=False, text=True)

elapsed = time.time() - start_time
hours = elapsed / 3600
minutes = (elapsed % 3600) / 60

print("\n" + "="*70)
if result.returncode == 0:
    print("✅ BENCHMARK SUITE COMPLETED SUCCESSFULLY")
else:
    print("⚠️  BENCHMARK SUITE COMPLETED WITH SOME ERRORS")
    print("   (Some experiments may have failed but results are still available)")
print("="*70)
print(f"⏱️  Total time: {hours:.1f} hours ({minutes:.1f} minutes)")
print("="*70)

In [ ]:
# Cell 6: Display Results Summary
import pandas as pd
from pathlib import Path

results_path = Path(RESULTS_DIR)

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

# Check if results directory exists
if not results_path.exists():
    print("\n⚠️  Results directory not found. Run Cell 5 first.")
else:
    # List all result files
    csv_files = list(results_path.rglob("*.csv"))
    print(f"\n📁 Found {len(csv_files)} result files:\n")
    
    if len(csv_files) == 0:
        print("   No CSV files found yet. Benchmark may still be running.")
    else:
        for f in sorted(csv_files)[:20]:  # Show first 20
            try:
                print(f"  {f.relative_to(results_path)}")
            except ValueError:
                print(f"  {f}")
        
        if len(csv_files) > 20:
            print(f"  ... and {len(csv_files) - 20} more")
    
    # Show cross-experiment aggregation if exists
    agg_file = results_path / "analysis" / "cross_experiment_aggregation.csv"
    if agg_file.exists():
        print("\n📊 Cross-Experiment Aggregation:")
        try:
            agg_df = pd.read_csv(agg_file)
            display(agg_df)
        except Exception as e:
            print(f"   Error reading aggregation file: {e}")
    
    # Show optimizer rankings if exists
    rank_file = results_path / "analysis" / "optimizer_rankings.csv"
    if rank_file.exists():
        print("\n🏆 Optimizer Rankings:")
        try:
            rank_df = pd.read_csv(rank_file)
            display(rank_df)
        except Exception as e:
            print(f"   Error reading rankings file: {e}")

In [ ]:
# Cell 7: Show Experiment-Specific Results

experiments_dir = results_path / "experiments"

if not experiments_dir.exists():
    print("⚠️  Experiments directory not found yet.")
else:
    found_experiments = False
    for exp_dir in sorted(experiments_dir.iterdir()):
        if exp_dir.is_dir():
            csv_files = list(exp_dir.glob("*.csv"))
            if csv_files:
                found_experiments = True
                print(f"\n{'='*60}")
                print(f"📁 {exp_dir.name.upper()} RESULTS")
                print(f"{'='*60}")
                
                # Try to load and display main results
                for csv_file in sorted(csv_files)[:3]:  # Show first 3
                    try:
                        df = pd.read_csv(csv_file)
                        print(f"\n📄 {csv_file.name}")
                        print(f"   Shape: {df.shape}")
                        if 'optimizer' in df.columns or 'Optimizer' in df.columns:
                            opt_col = 'optimizer' if 'optimizer' in df.columns else 'Optimizer'
                            print(f"   Optimizers: {df[opt_col].unique().tolist()}")
                        display(df.head())
                    except Exception as e:
                        print(f"   Error reading {csv_file.name}: {e}")
    
    if not found_experiments:
        print("   No experiment results found yet.")

In [ ]:
# Cell 8: Visualizations
import matplotlib.pyplot as plt

viz_dir = results_path / "visualizations"

if not viz_dir.exists():
    print("⚠️  Visualizations directory not found yet.")
else:
    # List available visualizations
    html_files = list(viz_dir.rglob("*.html"))
    png_files = list(viz_dir.rglob("*.png"))
    
    print(f"\n📈 Found {len(html_files)} interactive HTML plots")
    print(f"📊 Found {len(png_files)} static PNG plots")
    
    if len(png_files) > 0:
        # Display some PNG plots
        for png_file in sorted(png_files)[:6]:  # Show first 6
            try:
                img = plt.imread(str(png_file))
                fig, ax = plt.subplots(figsize=(10, 6))
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(png_file.stem, fontsize=12)
                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f"Could not display {png_file.name}: {e}")
    else:
        print("   No PNG visualizations found yet.")
    
    if len(html_files) > 0:
        print(f"\n   Interactive HTML plots saved to: {viz_dir}/interactive/")
        print("   (Download and open in browser to view)")

In [ ]:
# Cell 9: Statistical Analysis Summary

analysis_dir = results_path / "analysis"

if not analysis_dir.exists():
    print("⚠️  Analysis directory not found yet.")
else:
    print("=" * 60)
    print("STATISTICAL ANALYSIS")
    print("=" * 60)
    
    # Show cross-experiment statistics
    stats_file = analysis_dir / "cross_experiment_statistics.csv"
    if stats_file.exists():
        print("\n🔬 Cross-Experiment Statistical Comparisons:")
        try:
            stats_df = pd.read_csv(stats_file)
            display(stats_df)
        except Exception as e:
            print(f"   Error reading statistics file: {e}")
    else:
        print("\n   Cross-experiment statistics not generated yet.")
    
    # Show basic statistics
    basic_stats = analysis_dir / "00_basic_statistics.csv"
    if basic_stats.exists():
        print("\n📊 Basic Statistics:")
        try:
            basic_df = pd.read_csv(basic_stats)
            display(basic_df.head(20))
        except Exception as e:
            print(f"   Error reading basic statistics: {e}")
    else:
        print("\n   Basic statistics not generated yet.")

In [ ]:
# Cell 10: Archive Results for Download
import shutil
from datetime import datetime

if IS_KAGGLE:
    try:
        archive_name = f"gdsearch_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        archive_path = f"/kaggle/working/{archive_name}"
        
        print(f"Creating archive: {archive_name}.zip")
        
        # Check if results directory has content
        if results_path.exists() and any(results_path.iterdir()):
            shutil.make_archive(archive_path, 'zip', RESULTS_DIR)
            print(f"\n✅ Results archived to: {archive_path}.zip")
            print("   Download from Kaggle Output tab")
        else:
            print("\n⚠️  No results to archive yet. Run Cell 5 first.")
    except Exception as e:
        print(f"\n❌ Error creating archive: {e}")
else:
    print(f"Not on Kaggle - results saved to: {RESULTS_DIR}")

In [ ]:
# Cell 12: Quick Access Guide

print("""
=================================================================
📖 RESULTS QUICK ACCESS GUIDE
=================================================================

📁 Results Directory Structure:
   results/
   ├── experiments/           # Individual experiment data
   │   ├── mnist/            # MNIST benchmark results
   │   ├── cifar10/          # CIFAR-10 results
   │   ├── nlp/              # NLP (IMDB) results
   │   ├── resnet/           # ResNet18 results
   │   ├── highdim/          # High-dimensional functions
   │   ├── 2d/               # 2D test functions
   │   ├── robustness/       # Robustness analysis
   │   ├── sam/              # SAM sensitivity analysis
   │   └── ablation/         # Ablation studies
   │
   ├── analysis/              # Statistical analyses
   │   ├── cross_experiment_aggregation.csv    # Combined results
   │   ├── optimizer_rankings.csv              # Overall rankings
   │   ├── cross_experiment_statistics.csv     # Statistical tests
   │   ├── 00_basic_statistics.csv             # Basic stats
   │   ├── 01_convergence_rates.csv            # Convergence analysis
   │   └── 02_statistical_comparison.csv       # Pairwise comparisons
   │
   ├── visualizations/        # Plots and visualizations
   │   ├── interactive/       # Interactive HTML plots
   │   └── static/            # Static PNG/PDF plots
   │
   └── reports/               # Summary reports
       ├── experiment_summary_report.md
       └── 00_EXPERIMENT_SUMMARY.md

=================================================================

📊 Key Result Files:
   1. Cross-experiment aggregation:
      → analysis/cross_experiment_aggregation.csv
      
   2. Optimizer rankings (sorted by performance):
      → analysis/optimizer_rankings.csv
      
   3. Statistical significance tests:
      → analysis/cross_experiment_statistics.csv
      
   4. Convergence analysis:
      → analysis/01_convergence_rates.csv

=================================================================

🔬 Statistical Analysis Features:
   ✅ Multi-seed experiments (10 seeds for high statistical power)
   ✅ Student's t-tests for significance
   ✅ Cohen's d effect sizes
   ✅ Statistical power analysis
   ✅ Multiple comparison corrections:
      - Holm-Bonferroni (FWER control)
      - Benjamini-Hochberg (FDR control)

=================================================================

🔄 Resume Logic:
   • Uses --resume flag to skip completed experiments
   • Checks for existing CSV files before running
   • Safe to interrupt and restart at any time
   • Progress is saved incrementally

=================================================================

📈 Visualizations Available:
   • Training/test loss curves
   • Accuracy progression plots
   • Loss landscape 3D surfaces
   • Statistical comparison heatmaps
   • Convergence rate comparisons
   • Interactive parameter sensitivity plots

=================================================================

💾 Download Results:
   Kaggle: Results archived to .zip in /kaggle/working/
           Download from Output tab after completion
   
   Local: Results saved to: {RESULTS_DIR}

=================================================================

📖 For detailed documentation, see:
   • README.md in results directory
   • reports/experiment_summary_report.md
   • Individual experiment folders for per-run details

=================================================================
""".format(RESULTS_DIR=RESULTS_DIR))